In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(
    "",
    connect_args={"sslmode": "require"}
)

In [3]:
df = pd.read_sql(
    'SELECT * FROM "Amount_of_collected_municipal_wastes";',
    engine
)
df

,Id,Year,Total,Paper,Glass,Plastic,Metal_iron_steel_aluminum,Organic_waste_food_leaves,Textile,Rubber,Mixed_municipal_waste,Other
0,1,2014,569794.0,10830.0,2520.0,6735.0,2006.0,38017.0,10078.0,336.0,487711.0,11560.0
1,2,2015,620328.0,9386.0,2150.0,6899.0,2186.0,36891.0,8652.0,1395.0,533721.0,19049.0
2,3,2016,610227.0,11293.0,1850.0,10197.0,2125.0,25240.0,7943.0,656.0,543644.0,7281.0
3,4,2017,635875.0,8740.0,2208.0,9262.0,1949.0,38411.0,8181.0,1007.0,554280.0,11838.0
4,5,2018,625386.0,7729.0,2288.0,8532.0,1991.0,35756.0,7837.0,673.0,535604.0,24975.0
5,6,2019,632484.0,9110.0,2726.0,9171.0,1656.0,38139.0,6252.0,778.0,542664.0,21988.0
6,7,2020,630086.0,10513.0,3454.0,11630.0,1946.0,39983.0,10112.0,1005.0,525745.0,25698.0
7,8,2021,632087.0,12263.0,3713.0,12871.0,2241.0,41687.0,8737.0,1598.0,522914.0,26063.0
8,9,2022,605638.0,12178.0,3754.0,13063.0,2302.0,40259.0,8373.0,1487.0,494693.0,29531.0
9,10,2023,621686.0,10914.0,3397.0,14532.0,2203.0,39690.0,6695.0,1301.0,531033.0,11920.0


In [4]:
df = df.drop("Id", axis=1)

In [5]:
df['Year'] = df['Year'].astype(int)

columns_to_predict = [col for col in df.columns if col != 'Year']

lagged_df = df.copy()
for col in columns_to_predict:
    lagged_df[f'{col}_lag1'] = lagged_df[col].shift(1)

lagged_df = lagged_df.dropna().reset_index(drop=True)

X_cols = [f'{col}_lag1' for col in columns_to_predict]

In [6]:
from sklearn.linear_model import LinearRegression

models = {}
for target in columns_to_predict:
    X = lagged_df[X_cols]
    y = lagged_df[target]

    model = LinearRegression()
    model.fit(X, y)
    models[target] = model

In [7]:
last_row = df[df['Year'] == 2023][columns_to_predict].values.flatten()

future_years = [2024, 2025, 2026, 2027]
predictions = {}

for year in future_years:
    X_input = pd.DataFrame([last_row], columns=X_cols)

    pred_row = []
    for i, target in enumerate(columns_to_predict):
        model = models[target]
        pred_value = model.predict(X_input)[0]
        pred_row.append(pred_value)

    predictions[year] = dict(zip(columns_to_predict, pred_row))

    last_row = pred_row

pred_df = pd.DataFrame(predictions).T
pred_df.index.name = 'Year'
pred_df

,Total,Paper,Glass,Plastic,Metal_iron_steel_aluminum,Organic_waste_food_leaves,Textile,Rubber,Mixed_municipal_waste,Other
Year,,,,,,,,,,
2024,637471.000000,12347.000000,3334.000000,16852.000000,2435.000000,33175.000000,6189.000000,1136.000000,546866.000000,15139.000000
2025,622194.931548,10428.866970,4075.773577,13065.984545,1987.739633,47048.992191,5322.345123,1415.177700,499618.371239,39235.962806
2026,617216.594406,8283.360550,4218.994227,13753.393776,1811.837050,47669.154329,6953.420754,975.252883,505250.735416,28296.895462
2027,662614.019276,15072.696548,4267.621345,21131.201972,2436.112352,33414.953442,6045.269112,1330.897599,568974.768535,9940.015778


In [8]:
import joblib
from pathlib import Path
import json

MODEL_DIR = Path("../models/Amount_of_collected_municipal_wastes")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for target, model in models.items():
    joblib.dump(model, MODEL_DIR / f"{target}.pkl")

metadata = {
    "dataset": "Amount_of_collected_municipal_wastes",
    "model_type": "LinearRegression",
    "lag": 1,
    "trained_until_year": int(df["Year"].max()),
    "features": X_cols,
    "targets": columns_to_predict
}

with open(MODEL_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

In [ ]:
df = pd.read_sql(
    'SELECT * FROM "Collected_and_generated_municipal_wastes";',
    engine
)
df